In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. DONNÉES DU SYSTÈME (Basées sur le rapport) ---
TN = 3.3          # Couple nominal [Nm]
IN = 4.5          # Courant nominal [A] efficace
U12 = 45.0        # Tension induite [V] efficace à 1000 rpm
R = 2.7           # Résistance de phase [Ohm]
L = 9e-3          # Inductance de phase [H]
Jm = 0.24e-3      # Inertie du rotor [kg*m^2]
Jch = 1.62e-4     # Inertie de la charge (bague laiton calculée) [kg*m^2]

# Retards identifiés par Sysquake
Tmi_mes = 160e-6  # Constante de temps mesure courant [s]
Tr_mes = 132e-6   # Retard pur de commande [s]

# --- 2. CALCULS DES CONSTANTES DE TEMPS ---
Jt = Jm + Jch     # Inertie totale
omegaNom = (1000.0 / 60.0) * 2.0 * np.pi

# Constantes de tension (ke) et de couple (kt) en valeurs crêtes et par phase
ke = (U12 * np.sqrt(2) / np.sqrt(3)) / omegaNom
kt = TN / (np.sqrt(2) * IN)

Te = L / R
Tm = (Jt * R) / (kt * ke)
Kth = Tm / R
Kpratique = Kth * 1.01  # Gain adapté pratiquement (Kai)

# Points demandés par l'énoncé :
tau_dom = np.sqrt(Te * Tm)
sigma_tau_pct = Tr_mes + Tmi_mes
N_dom = int(np.ceil(tau_dom / sigma_tau_pct))

print(f"--- Caractéristiques du système ---")
print(f"τ_dom (Constante dominante) : {tau_dom*1e3:.2f} ms")
print(f"Σ τ_pct (Somme des petites constantes) : {sigma_tau_pct*1e6:.2f} µs")
print(f"N_dom maximal calculé : {N_dom}\n")

# --- 3. BALAYAGE DE N (1 à 15) ---
N_values = np.arange(1, 16)
w = np.logspace(1, 5, 10000)  # Vecteur de fréquences pulsatiles (rad/s)
s = 1j * w

# Réponse fréquentielle intrinsèque de la fonction de transfert Gai(s)
# Gai(s) = Kpratique * [s / (1 + s*Tm + s^2*Te*Tm)] * [1 / (1 + s*Tmi_mes)] * exp(-s*Tr_mes)
Gai_freq = Kpratique * (s / (1.0 + s * Tm + (s**2) * Tm * Te)) * (1.0 / (1.0 + s * Tmi_mes)) * np.exp(-s * Tr_mes)

# Initialisation des tableaux de résultats
Kpi_45, ES_45, w_45 = [], [], []
Kpi_60, ES_60, w_60 = [], [], []

for N in N_values:
    Tii = N * sigma_tau_pct
    # Régulateur PI avec Kpi = 1 pour trouver la phase
    Gri_unitaire = (1.0 + s * Tii) / (s * Tii)
    Gbo_unitaire = Gri_unitaire * Gai_freq
    
    phase_Gbo = np.angle(Gbo_unitaire, deg=True)
    # Ajustement de la phase entre -360° et 0°
    phase_Gbo = (phase_Gbo + 180) % 360 - 180
    phase_Gbo[phase_Gbo > 0] -= 360

    # --- Cas Marge de Phase = 45° (Phase requise = -135°) ---
    idx_45 = np.argmin(np.abs(phase_Gbo + 135))
    w_c45 = w[idx_45]
    gain_db45 = 20 * np.log10(np.abs(Gbo_unitaire[idx_45]))
    kpi45_val = 10**(-gain_db45 / 20)
    gbo0_45 = (kpi45_val * Tm) / (R * N * sigma_tau_pct)
    es45_val = 1.0 / (1.0 + gbo0_45)
    
    Kpi_45.append(kpi45_val)
    w_45.append(w_c45)
    ES_45.append(es45_val * 100)

    # --- Cas Marge de Phase = 60° (Phase requise = -120°) ---
    idx_60 = np.argmin(np.abs(phase_Gbo + 120))
    w_c60 = w[idx_60]
    gain_db60 = 20 * np.log10(np.abs(Gbo_unitaire[idx_60]))
    kpi60_val = 10**(-gain_db60 / 20)
    gbo0_60 = (kpi60_val * Tm) / (R * N * sigma_tau_pct)
    es60_val = 1.0 / (1.0 + gbo0_60)
    
    Kpi_60.append(kpi60_val)
    w_60.append(w_c60)
    ES_60.append(es60_val * 100)

# --- 4. AFFICHAGE DES VALEURS RETENUES (COMPROMIS) ---
# N=6 pour 45° et N=9 pour 60° selon les conclusions du rapport
print(f"--- Résultats pour le compromis N=6 (Marge 45°) ---")
print(f"Kpi : {Kpi_45[5]:.2f} | Tii : {6*sigma_tau_pct*1e3:.3f} ms | Gi_i (Kpi/Tii) : {Kpi_45[5]/(6*sigma_tau_pct):.2f}")
print(f"Erreur statique : {ES_45[5]:.2f} % | Bande passante : {w_45[5]:.0f} rad/s")

print(f"\n--- Résultats pour le compromis N=9 (Marge 60°) ---")
print(f"Kpi : {Kpi_60[8]:.2f} | Tii : {9*sigma_tau_pct*1e3:.3f} ms | Gi_i (Kpi/Tii) : {Kpi_60[8]/(9*sigma_tau_pct):.2f}")
print(f"Erreur statique : {ES_60[8]:.2f} % | Bande passante : {w_60[8]:.0f} rad/s\n")

# --- 5. GÉNÉRATION DES GRAPHES ---
plt.figure(figsize=(12, 10))

# Graphe pour la Marge de Phase de 45°
plt.subplot(2, 1, 1)
ax1 = plt.gca()
color = 'tab:blue'
ax1.set_xlabel('Valeur de N')
ax1.set_ylabel('Erreur statique (%)', color=color)
line1 = ax1.plot(N_values, ES_45, 'o-', color=color, label='Erreur statique')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True)

ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Bande passante (rad/s)', color=color)
line2 = ax2.plot(N_values, w_45, 's-', color=color, label='Bande passante')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Marge de phase 45° : Évolution de la Bande Passante et Erreur Statique')
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='center left')

# Graphe pour la Marge de Phase de 60°
plt.subplot(2, 1, 2)
ax3 = plt.gca()
color = 'tab:blue'
ax3.set_xlabel('Valeur de N')
ax3.set_ylabel('Erreur statique (%)', color=color)
line3 = ax3.plot(N_values, ES_60, 'o-', color=color, label='Erreur statique')
ax3.tick_params(axis='y', labelcolor=color)
ax3.grid(True)

ax4 = ax3.twinx()
color = 'tab:orange'
ax4.set_ylabel('Bande passante (rad/s)', color=color)
line4 = ax4.plot(N_values, w_60, 's-', color=color, label='Bande passante')
ax4.tick_params(axis='y', labelcolor=color)

plt.title('Marge de phase 60° : Évolution de la Bande Passante et Erreur Statique')
lines2 = line3 + line4
labels2 = [l.get_label() for l in lines2]
ax3.legend(lines2, labels2, loc='center left')

plt.tight_layout()
plt.show()
